# Phase 3: Smart Hybrid Retrieval-Augmented N-Gram Chatbot
**Architecture**: Turn-Pair Inverted Indexing (BM25/TF-IDF) + N-Gram Hybrid Generation + Multi-Criteria Re-Ranking
**Corpus**: DailyDialog (76,052 prompt-reply dialogue turns)

---

## 1. Problem & Architecture Overview
In classical N-gram generation, queries like *"I want to buy a computer"* fail because:
1. The bot echoes the verb (*"Want to the company."*).
2. Follow-ups like *"How much does it cost?"* stitch broken fragments (*"Computer and I will be a good."*).

### The Smart Hybrid Solution:
```
User Query -> Coreference Memory -> Inverted Index (76k pairs) -> Top Candidates
                                                                      │
     ┌────────────────────────────────────────────────────────────────┘
     ▼
[Channel A: Authentic Dialogue Turn] + [Channel B: N-Gram Completion from Opener]
     ▼
Multi-Criteria Scoring: Score = 0.4*Match + 3.5*TopicOverlap + LM_Fluency - Penalties
     ▼
Top-Scoring Natural Response
```


In [1]:
import sys
from pathlib import Path
root = Path.cwd()
while root.name and not (root / "src").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.chatbot import SmartChatbot

bot = SmartChatbot(corpus_name="dailydialog", n=3, temperature=0.7, seed=42)
print("SmartChatbot initialized.")
print(f"Total Dialogue Pairs Indexed: {len(bot.dialogue_index.pairs):,}")


SmartChatbot initialized.
Total Dialogue Pairs Indexed: 76,052


## 2. Resolving Real-World Queries (Screenshot Test Cases)


In [1]:
queries = [
    "Hello! How are you doing today?",
    "I want to buy a computer.",            # Addresses Turn 1 failure
    "How much does it cost?",               # Addresses Turn 2 failure ('it' -> computer)
    "Can you recommend a good laptop?",
    "Thank you for your help!",
]

for i, q in enumerate(queries, 1):
    reply = bot.respond(q)
    print(f"Turn {i}:")
    print(f"  User : {q}")
    print(f"  Smart: {reply}")
    print(f"  [Topic Memory]: {bot.last_topic}\n")


Turn 1:
  User : Hello! How are you doing today?
  Smart: Hello! Nice to meet you. How are you today?
  [Topic Memory]: None

Turn 2:
  User : I want to buy a computer.
  Smart: Well, you heard right.You can get an excellent deal on a new computer here.We have great discounts for students.
  [Topic Memory]: computer

Turn 3:
  User : How much does it cost?
  Smart: It costs 125 dollars a month.
  [Topic Memory]: cost

Turn 4:
  User : Can you recommend a good laptop?
  Smart: What about the thinkpad? It is lightweight and very reliable.
  [Topic Memory]: laptop

Turn 5:
  User : Thank you for your help!
  Smart: You're welcome!
  [Topic Memory]: help


## 3. Comparison with Earlier Bots
| Query | Keyword Bot | Smart Hybrid Bot |
|:---|:---|:---|
| *"I want to buy a computer"* | *"Want to the company."* | *"Well, you heard right. You can get an excellent deal on a new computer here."* |
| *"How much does it cost?"* | *"Computer and I will be a good."* | *"It costs 125 dollars a month."* |

The hybrid architecture successfully combines the statistical generative capabilities of N-Grams with the communicative authenticity of human dialogue.
